In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm.notebook import tqdm
from hmmlearn import hmm, vhmm

from synthetic_data_generation_functions import *
from synthetic_data_analysis_functions import *
from hmm_functions import *


plt.style.use('../../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


QSocketNotifier: Can only be used with threads started with QThread


In [14]:
def reformat_hmm(synthetic_data, model):

    nbr_of_states = len(model.transmat_)

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    initial_state_list = []
    sequences_number = len(test_data)

    for i in range(sequences_number):
        
        choices_sequence = test_data[i]
        
        states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
        initial_state_list.append(states_sequence[0])

    initial_state_list_distri = []

    for s in range(len(model.transmat_)):

        initial_state_list_distri.append(initial_state_list.count(s))

    transmat = model.transmat_
    emission_vect = model.emissionprob_[:,2]
    mat = transmat
    sorted_indexes = np.argsort(emission_vect)

    ##

    new_transmat = order_matrix(mat, sorted_indexes)

    ##

    new_emissionmat = []
    new_initial_state_list_distri = []

    for i in sorted_indexes:
        new_emissionmat.append(model.emissionprob_[i,:])
        new_initial_state_list_distri.append(initial_state_list_distri[i])

    new_emissionmat = np.array(new_emissionmat)
    new_initial_state_distri = np.array(new_initial_state_list_distri)/np.sum(new_initial_state_list_distri)

    return {'emissionmat':new_emissionmat, 'initial_state_distri':new_initial_state_distri, 'transmat':new_transmat}

# Simulations generations

## Parameters

In [3]:
p_cw_reward = 0.8
p_ccw_reward = 0
p_cw_init = 0.5
steps_number = 100
noise_amplitude = 0.1
drift_matrix = np.array([[ 0.05 , -0.05],
                         [ 0    , 0   ]])
drift_init = 0.0

args_dict = {'p_cw_reward': p_cw_reward, 'p_ccw_reward': p_ccw_reward, 'p_cw_init': p_cw_init, 'steps_number': steps_number, 'noise_amplitude': noise_amplitude, 'drift_matrix': drift_matrix, 'drift_init': drift_init}

number_of_simulations_perset = 200

n_simulations_list = [number_of_simulations_perset]*100

start_index = 0

# fit_type = 'aic'
fit_type = 'score'

simulations_folder_path = f'/home/david/Documents/code/DDM_v6_synthetic_data_identical_drifts_{fit_type}_parallel'


## Generation

In [9]:

generate_simulations = False # PARAMS


for i, n_simulations in enumerate(n_simulations_list):
    
    p_cw_init_range = np.linspace(0,1,100)
    
    if not(generate_simulations):

        break

    
    simulations_batch = run_simulations_batch_random_init(args_dict, n_simulations, p_cw_init_range)

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + i+1}.pkl', 'wb') as file:
        pickle.dump(simulations_batch, file)

# HMM fit

## Parameters

In [5]:
n_to_test = np.arange(2,15)
expected_time = np.ceil(len(n_simulations_list)/jl.cpu_count())
print(f'Expected time : {int(expected_time)} t_job')

Expected time : 13 t_job


## Model fit

In [6]:
def reformat_synthetic_data(synthetic_data):

    slice_size = int(len(synthetic_data)/2)

    # synthetic_data = synthetic_data[:int(slice_size/2)] + synthetic_data[int(slice_size):int(3*slice_size/2)] + synthetic_data[int(slice_size/2):int(slice_size)] + synthetic_data[int(3*slice_size/2):]

    training_qts_sequences = [synthetic_data[i]['choices'] for i in np.arange(0,slice_size)]
    validation_qts_sequences = [synthetic_data[i]['choices'] for i in np.arange(slice_size,2*slice_size)]
    training_rewards_sequences = [synthetic_data[i]['rewards'] for i in np.arange(0,slice_size)]
    validation_rewards_sequences = [synthetic_data[i]['rewards'] for i in np.arange(slice_size,2*slice_size)]

    # action_types = np.array([[0, 1], # CCW Unrewarded, CCW Rewarded
    #                          [2, 3]]) # CW Unrewarded, CW Rewarded

    action_types = np.arange(3) # CCW, U; CW, U; CW, R

    training_sequences = []

    for training_qts_sequence, training_rewards_sequence in zip(training_qts_sequences, training_rewards_sequences):

        training_sequences.append([])

        for qt, reward in zip(training_qts_sequence, training_rewards_sequence):

            training_sequences[-1].append(action_types[qt + reward])


    validation_sequences = []

    for validation_qts_sequence, validation_rewards_sequence in zip(validation_qts_sequences, validation_rewards_sequences):

        validation_sequences.append([])

        for qt, reward in zip(validation_qts_sequence, validation_rewards_sequence):

            validation_sequences[-1].append(action_types[qt + reward])

    ###################
    ### Infer model ###
    ###################

    ## Reformating data

    reformated_training_sequences = np.array([]).astype(int)
    reformated_validation_sequences = np.array([]).astype(int)

    for x,y in zip(training_sequences,validation_sequences):

        reformated_training_sequences = np.concatenate((reformated_training_sequences, x))
        reformated_validation_sequences = np.concatenate((reformated_validation_sequences, y))

    reformated_training_sequences = reformated_training_sequences.reshape(-1,1)
    reformated_training_sequences_lengths = [len(x) for x in training_sequences]

    reformated_validation_sequences = reformated_validation_sequences.reshape(-1,1)
    reformated_validation_sequences_lengths = [len(y) for y in validation_sequences]

    return reformated_training_sequences, reformated_validation_sequences, reformated_training_sequences_lengths, reformated_validation_sequences_lengths



def fit_hmm(index, n_simulations, fit_type='score', save_path=None):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{index}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    ########################
    ### Reformating Data ###
    ########################

    reformated_training_sequences, reformated_validation_sequences, reformated_training_sequences_lengths, reformated_validation_sequences_lengths = reformat_synthetic_data(synthetic_data)

    ###################
    ### Infer model ###
    ###################

    if fit_type == 'score':

        res = infer_best_model_score(reformated_training_sequences, reformated_validation_sequences, 
                                                reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                n_to_test, n_fits = 200, save_path=save_path)
            
        return res
    
    elif fit_type=='aic':

        best_model, best_score = infer_best_model_aic(reformated_training_sequences, reformated_validation_sequences, 
                                                reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                n_to_test, n_fits = 200)
            
        return best_model            


In [7]:
# fit_model = True # PARAM

# if fit_model:

#     best_models_list = jl.Parallel(n_jobs=-1)(jl.delayed(fit_hmm)(start_index+i+1, number_of_simulations_perset) for i in range(len(n_simulations_list)))

#     for index, best_model in enumerate(best_models_list):
    
#         if fit_type=='score':
                
#             with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/best_model_score_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'wb') as file:
#                 pickle.dump(best_model, file)

#         elif fit_type=='aic':
                
#             with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/best_model_aic_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'wb') as file:
#                 pickle.dump(best_model, file)



In [6]:
fit_model = False # PARAM

if fit_model:

    fit_output = jl.Parallel(n_jobs=-2)(jl.delayed(fit_hmm)(start_index+i+1, 
                                                            number_of_simulations_perset, 
                                                            save_path=f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl')
                                                            for i in tqdm(range(len(n_simulations_list))))

    # fit_output = jl.Parallel(n_jobs=-2)(jl.delayed(fit_hmm)(start_index+i+1, number_of_simulations_perset) for i in tqdm(range(len(n_simulations_list))))

    # for i in range(len(fit_output)):
    
    #     if fit_type=='score':
                
    #         with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl', 'wb') as file:
    #             pickle.dump(fit_output[i], file)

else:

    fit_output = []

    for i in range(len(n_simulations_list)):
    
        with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl', 'rb') as file:
            batch_fit_output = pickle.load(file)

        fit_output.append(batch_fit_output)


In [7]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

best_states_nbr_list = []

for i in range(len(fit_output)):

    ax1.plot(n_to_test, fit_output[i][3], alpha=0.2)
    # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)

    best_states_nbr_list.append(len(fit_output[i][0].transmat_))

ax1.set_xlim([1,n_to_test[-1]+1])
ax1.set_xlabel("Number of states")
ax1.set_ylabel("log fit score")

ax2 = plt.subplot(row[1,0])

ax2.hist(best_states_nbr_list, bins=n_to_test, align='left')
ax2.set_xlim([1,n_to_test[-1]+1])
ax2.set_xlabel('Number of states')
ax2.set_ylabel('Number of HMM')


Text(0, 0.5, 'Number of HMM')

In [8]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

stacked_transmats = []
nbr_of_states = 9

for i in range(len(fit_output)):

    with open(f'{simulations_folder_path}/n_{n_simulations_list[0]}/simulations_batch_{n_simulations_list[0]}_test_{i+1}.pkl', 'rb') as file:
        synthetic_data = dill.load(file)

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    reformated_hmm = reformat_hmm(synthetic_data, fit_output[i][2][nbr_of_states-2])
    reformated_emissionmat = reformated_hmm['emissionmat']
    reformated_transmat = reformated_hmm['transmat']
    # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)
    stacked_transmats.append(reformated_transmat)

avrg_transmat = np.mean(stacked_transmats, axis=0)
std_transmat = np.std(stacked_transmats, axis=0)
var_transmat = np.var(stacked_transmats, axis=0)

ax1.imshow(avrg_transmat, vmin=0, vmax=1)
ax1.set_title('Average matrix')
ax1.set_xticks(np.arange(len(reformated_transmat)))
ax1.set_yticks(np.arange(len(reformated_transmat)))

ax2 = plt.subplot(row[1,0])
ax2.imshow(var_transmat, vmin=0, vmax=1)
ax2.set_title('Variance matrix')
ax2.set_xticks(np.arange(len(reformated_transmat)))
ax2.set_yticks(np.arange(len(reformated_transmat)))

# ax2.imshow(reformated_transmat, vmin=0, vmax=1)

# ax1.set_xlim([1,None])
# ax1.set_xlabel("Number of states")
# ax1.set_ylabel("log fit score")


# ax2.hist(best_states_nbr_list, bins=n_to_test, align='left')
# ax2.set_xlim([1,None])

In [9]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 2, hspace=0.5)

ax1 = plt.subplot(row[0,0])

batch_to_plot = 2
reformated_hmm = reformat_hmm(synthetic_data, fit_output[batch_to_plot][2][nbr_of_states-2])
reformated_emissionmat = reformated_hmm['emissionmat']
reformated_transmat = reformated_hmm['transmat']

is_best_model = len(reformated_transmat) == len(fit_output[batch_to_plot][0].transmat_)

ax1.imshow(reformated_transmat, vmin=0, vmax=1)

ax1.set_xlabel('To state')
ax1.set_ylabel('From state')
ax1.set_title(f'Batch {batch_to_plot}, {nbr_of_states} states {'*' if is_best_model else ''}')
# ax2 = plt.subplot(row[1,0])
# ax2.imshow(std_transmat, vmin=0, vmax=1)

ax2 = plt.subplot(row[0,1])
ax2.imshow(reformated_emissionmat, vmin=0, vmax=1)
ax2.set_ylabel('State')
ax2.set_xticks([0,1,2], labels=['CCW, U', 'CW, U','CW, R'], rotation=30, ha="right", rotation_mode="anchor")


In [ ]:
++++++++++++++++
+++++++++++++++++
++++++++++++++++++
+++++++++++++++++++
++++++++++++++++++++

SyntaxError: invalid syntax (2643378534.py, line 1)

In [28]:
with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/simulations_batch_{number_of_simulations_perset}_test_{2}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

reformated_training_sequences, reformated_validation_sequences, reformated_training_sequences_lengths, reformated_validation_sequences_lengths = reformat_synthetic_data(synthetic_data)

def infer_best_model_score_parallel(x_train, x_validate, training_lengths, validation_lengths, n_to_test, n_features = None, n_fits=200, n_jobs=-2):
    # check optimal score

    models_and_scores_list = jl.Parallel(n_jobs=n_jobs)(jl.delayed(fit_hmm_fixed_state_number)(x_train, x_validate, training_lengths, validation_lengths, n_states, n_fits=n_fits, n_features=n_features) for n_states in n_to_test)

    return models_and_scores_list

models_list = infer_best_model_score_parallel(reformated_training_sequences, reformated_validation_sequences, 
                                              reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                              np.arange(2,10), n_features = None, n_fits=200, n_jobs=-2)


In [30]:
fig=plt.figure(figsize=(1, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

for i,model in enumerate(models_list):

    ax1.scatter(np.arange(2,10)[i],model[1])

ax1.set_xlabel('number of states')
ax1.set_ylabel('log score')

ax1.set_xticks(np.arange(2,10))

In [ ]:
++++++++++++++++++

# Drift Estimation

## Generating "theoretical" average probability sequences

In [ ]:
# generate_theoretical_sequences = False # PARAM

# if generate_theoretical_sequences:

#     with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + 0+1}.pkl', 'rb') as file:
#             synthetic_data = pickle.load(file)

#     args = [synthetic_data[0]['parameters']] + [5000]
#     drift_range = np.linspace(0.01,0.2,200)

#     # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
#     test_msd_sequence_list = generate_test_msd_sequences_identical_drifts(drift_range, args)

#     with open(f'{simulations_folder_path}/test_msd_sequences.pkl', 'wb') as file:
#         pickle.dump([drift_range, test_msd_sequence_list], file)

# else:

#     with open(f'{simulations_folder_path}/test_msd_sequences.pkl', 'rb') as file:
#         drift_range, test_msd_sequence_list = pickle.load(file)


In [11]:
generate_theoretical_sequences = True # PARAM

if generate_theoretical_sequences:

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index +1}.pkl', 'rb') as file:
                synthetic_data = pickle.load(file)

    args = [synthetic_data[0]['parameters']] + [5000] + [p_cw_init_range]
    drift_range = np.linspace(0.01,0.2,200)

    # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
    test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts_random_init(drift_range, args)

    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'wb') as file:
        pickle.dump([drift_range, test_average_probability_sequences], file)

else:
 
    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'rb') as file:
        drift_range, test_average_probability_sequences = pickle.load(file)


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

...

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

## Minimizing mean square error

In [ ]:
save_result = False # PARAM
average_proba_sequences_hmm = []
msd_sequences_hmm = []
nbr_of_states = []
ordered_action_matrixes = []

fit_type = 'score' # 'score'

# for index, _ in enumerate(tqdm(n_simulations_list)):
for index, _ in enumerate(n_simulations_list):

    ##############################
    ### Loading Data and Model ###
    ##############################

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'rb') as file:
        batch_fit_output = pickle.load(file)

    model = batch_fit_output[0]

    nbr_of_states.append(len(model.transmat_))

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    """    initial_state_list = []
    sequences_number = len(test_data)

    for i in range(sequences_number):
        
        choices_sequence = test_data[i]
        
        states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
        initial_state_list.append(states_sequence[0])

    initial_state_list_distri = []

    for s in range(len(model.transmat_)):

        initial_state_list_distri.append(initial_state_list.count(s))

    transmat = model.transmat_
    emission_vect = model.emissionprob_[:,1]
    mat = transmat
    sorted_indexes = np.argsort(emission_vect)
    vector = np.ones([len(transmat),1])/len(transmat)

    ##

    new_transmat = order_matrix(mat, sorted_indexes)

    ##

    new_emissionmat = []
    new_initial_state_list_distri = []

    for i in sorted_indexes:
        new_emissionmat.append(model.emissionprob_[i,:])
        new_initial_state_list_distri.append(initial_state_list_distri[i])

    new_emissionmat = np.array(new_emissionmat)
    ordered_action_matrixes.append(new_emissionmat)
    new_initial_state_list_distri = np.array(new_initial_state_list_distri)/np.sum(new_initial_state_list_distri)"""

    ####################
    ### Computations ###
    ####################

    ## Probability sequences inference

    reconstructed_proba_sequences = infer_probability_sequence(model, test_data)

    stacked_proba_sequence = 0

    for action_sequence in test_data:

        reconstructed_proba_sequence = np.array(compute_reconstructed_proba_sequence(action_sequence, model))

        stacked_proba_sequence += reconstructed_proba_sequence

    average_proba_sequence_hmm = stacked_proba_sequence/len(test_data)

    average_proba_sequences_hmm.append(average_proba_sequence_hmm)

    mse_list = []

    # for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
    for test_average_probability_sequence in test_average_probability_sequences:
    
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

    if not(save_result):

        continue

    ##############################
    ### Saving Recovered drift ###
    ##############################
    
    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'wb') as file:
        pickle.dump([test_average_probability_sequences,recovered_drift], file)


# Mean square error analysis

## Recovered drift loading

In [17]:
recovered_drift_list = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'rb') as file:
        _, recovered_drift = pickle.load(file)
    
    recovered_drift_list.append(recovered_drift[0])

# Plots

In [ ]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51), density=True)
bin_width = histo[1][1] - histo[1][0]
ax1.stairs(histo[0]/np.sum(histo[0])/bin_width, histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

x = np.linspace(-0.2,0.2)
sigma = 0.1
ax1.plot(x, np.exp(-x**2/(2*sigma**2)) * 1/np.sqrt(2*sigma**2*np.pi), label = "Noise probability density")

ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Probability density')

ax1.legend()

NameError: name 'recovered_drift_list' is not defined

In [18]:
fig=plt.figure(figsize=(1, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51), density=False)
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True)
ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Number of simulations sets')

Text(0, 0.5, 'Number of simulations sets')

In [15]:
def compute_reconstructed_proba_sequence(choices_sequence, model):

    states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
    # print(states_sequence)

    emissionprob = model.emissionprob_

    reconstructed_p_cw_sequence = []

    for s in states_sequence:

        reconstructed_p_cw_sequence.append(emissionprob[s][1]+emissionprob[s][2])

    return reconstructed_p_cw_sequence


batch_index = 1
simulation_index = 9

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{batch_index+1}.pkl', 'rb') as file:
    synthetic_data = dill.load(file)

with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'rb') as file:
    batch_fit_output = pickle.load(file)

model = batch_fit_output[0] 

# reformated_hmm = reformat_hmm(synthetic_data, model)

# reformated_emissionmat = reformated_hmm['emissionmat']
# reformated_initial_state_distri = reformated_hmm['initial_state_distri']
# reformated_transmat = reformated_hmm['transmat']

ddm_result = synthetic_data[simulation_index]

choices_sequence = ddm_result['choices']
p_cw_sequence = ddm_result['p_cw']
rewards_sequence = ddm_result['rewards']

# action_types = np.array([[0, 1], # CCW Unrewarded, CCW Rewarded
#                          [2, 3]]) # CW Unrewarded, CW Rewarded

action_types = np.arange(3) # CCW, U; CW, U; CW, R


action_sequence = np.array([])

for qt, reward in zip(choices_sequence, rewards_sequence):

    action_sequence = np.append(action_sequence, action_types[qt+reward])

reconstructed_proba_sequence = compute_reconstructed_proba_sequence(action_sequence, model)
steps = np.arange(len(choices_sequence))

cw_proba = model.emissionprob_[:,1] + model.emissionprob_[:,2]
# cw_proba = model.emissionprob_[:,2]

fig=plt.figure(figsize=(3, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax = plt.subplot(row[0,0])

colors = ['violet' if i==1 else 'purple' for i in rewards_sequence]
# ax.scatter(steps, p_cw_sequence, label='Proba. Sequence', color=colors, marker='+', alpha=0.5)

# ax.scatter(steps, reconstructed_proba_sequence, label='Infered Proba. Sequence', color='green', marker='+', alpha=0.5)
# ax.plot(steps, reconstructed_proba_sequence, color='green', marker='+', alpha=0.2)


for s in range(len(model.transmat_)):

    state_proba = cw_proba[s]

    ax.axhline(state_proba, color='k', alpha=0.3, linestyle='--', linewidth=0.5)

ax.set_title(f'From set of {number_of_simulations_perset} simulations of length {steps_number}')

ax.set_xlabel('Step')
ax.set_ylabel('P_n(CW)')
ax.set_xticks(np.arange(0,len(choices_sequence),10))
ax.set_ylim([-0.05,1.05])



(-0.05, 1.05)

In [ ]:
def compute_reconstructed_proba_sequence(choices_sequence, model):

    states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
    # print(states_sequence)

    emissionprob = model.emissionprob_

    reconstructed_p_cw_sequence = []

    for s in states_sequence:

        reconstructed_p_cw_sequence.append(emissionprob[s][1]+emissionprob[s][2])

    return reconstructed_p_cw_sequence

msd_recovered_drift_lists = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_score_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        model = pickle.load(file)

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    reformated_hmm = reformat_hmm(synthetic_data, model)

    reformated_emissionmat = reformated_hmm['emissionmat']
    reformated_initial_state_distri = reformated_hmm['initial_state_distri']
    reformated_transmat = reformated_hmm['transmat']

    steps = np.arange(len(test_data[0]))
    mat_i = reformated_transmat
    proba_vect = reformated_emissionmat[:,1] + reformated_emissionmat[:,2]

    hmm_theory_msd_sequence = []

    for i in steps:

        # mat_i_previous = mat_i
        mat_i = np.matmul(mat_i,reformated_transmat)
        #(np.matmul(reformated_initial_state_distri, mat_i) - np.matmul(reformated_initial_state_distri,mat_i_previous)) * reformated_emissionmat[:,1]
        
        term_1 = np.vecdot(np.vecmat(reformated_initial_state_distri, mat_i), proba_vect**2)
        term_2 = np.vecdot(reformated_initial_state_distri,proba_vect)**2
        term_3 = -2 * np.vecdot(np.vecmat(reformated_initial_state_distri, mat_i),proba_vect) * np.vecdot(reformated_initial_state_distri,proba_vect)

        msd = term_1 + term_2 + term_3

        hmm_theory_msd_sequence.append(msd)

    hmm_infered_msd_sequence = []
    stacked_reconstructed_proba_sequence = 0

    for seq in test_data:

        reconstructed_proba_sequence = compute_reconstructed_proba_sequence(seq,model)
        stacked_reconstructed_proba_sequence += (reconstructed_proba_sequence[0] - reconstructed_proba_sequence)**2

    hmm_infered_msd_sequence = stacked_reconstructed_proba_sequence/len(test_data)

    mse_list = []

    for test_msd_sequence in test_msd_sequence_list:
    
        mse_list.append(compute_mean_square_error_v2(hmm_infered_msd_sequence, test_msd_sequence))

    min_mse = np.min(mse_list)
    msd_recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]
    msd_recovered_drift_lists.append(msd_recovered_drift)

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(msd_recovered_drift_lists, bins=np.linspace(0.01,0.2,51), density=False)
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True)
ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Number of simulations sets')


Text(0, 0.5, 'Number of simulations sets')

In [ ]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 2, hspace=0.5, width_ratios = [1,0.2])

ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[0,1])
batch_index = 11

batch_to_plot = 0
nbr_of_states = 9

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = dill.load(file)

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'rb') as file:
        batch_fit_output = pickle.load(file)

    reformated_hmm = reformat_hmm(synthetic_data, batch_fit_output[2][nbr_of_states-2])
    # reformated_hmm = reformat_hmm(synthetic_data, batch_fit_output[0])

    reformated_emissionmat = reformated_hmm['emissionmat']
    reformated_transmat = reformated_hmm['transmat']

    proba_dist = reformated_emissionmat[:,2]
    
    ax1.plot(proba_dist, marker='s', markersize=2)
    ax2.scatter(0,np.mean(np.diff(reformated_emissionmat[:,2])), marker='_', alpha=0.5)

ax1.set_xlabel('State')
ax1.set_ylabel('Proba to do CW,R')
ax1.set_xticks(np.arange(0,n_to_test[-1]))

ax2.set_ylabel('Average slope')

# ax.set_title(f'Average slope : {np.mean(np.diff(new_emissionmat[:,2]))}')


Text(0, 0.5, 'Average slope')

In [15]:
def compute_hmm_slope(transmat, emissionmat):

    slope_list = []
    # proba_vect = emissionmat[:,1] + emissionmat[:,2]
    proba_vect = emissionmat[:,2]

    for i in range(len(transmat)):

        slope = np.sum(transmat[i,:] * proba_vect) - proba_vect[i]
        slope_list.append(slope)

    return slope_list

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1,1, hspace=0.5)

ax1 = plt.subplot(row[0,0])
batch_index = 11

batch_to_plot = 0
nbr_of_states = 9

for index, _ in enumerate(n_simulations_list[:10]):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = dill.load(file)

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + index+1}.pkl', 'rb') as file:
        batch_fit_output = pickle.load(file)

    reformated_hmm = reformat_hmm(synthetic_data, batch_fit_output[2][nbr_of_states-2])
    # reformated_hmm = reformat_hmm(synthetic_data, batch_fit_output[0])

    reformated_emissionmat = reformated_hmm['emissionmat']
    reformated_transmat = reformated_hmm['transmat']

    hmm_slope_list = compute_hmm_slope(reformated_transmat,reformated_emissionmat)
    
    ax1.plot(hmm_slope_list, marker='s', markersize=2, alpha=0.2)

ax1.set_xlabel('State')
ax1.set_ylabel('HMM P_(CW) slope')
ax1.set_xticks(np.arange(0,n_to_test[-1]))



# ax.set_title(f'Average slope : {np.mean(np.diff(new_emissionmat[:,2]))}')


In [62]:
hmm_slope_list

[array([5.85236207e-02, 9.55394215e-04, 4.41887833e-02, 9.74998415e-14,
        7.48382558e-04, 1.74813520e-18, 1.02837743e-07, 5.38784674e-02,
        7.61745477e-05]),
 array([1.44999688e-03, 2.43680730e-10, 3.11204624e-08, 4.79065377e-05,
        3.77999353e-01, 7.35054627e-03, 5.92915464e-05, 3.51399812e-05,
        1.68050120e-01]),
 array([7.09723580e-02, 2.00892450e-08, 2.51015743e-10, 8.59736816e-06,
        3.15827882e-16, 3.65315965e-02, 7.73026245e-10, 2.11179327e-03,
        2.33539212e-04]),
 array([2.48402466e-05, 7.94407297e-11, 5.31813234e-06, 7.69081731e-07,
        5.74386579e-06, 6.28644014e-08, 3.20810408e-01, 2.42853007e-01,
        2.24098293e-01]),
 array([6.17798680e-04, 4.86572646e-02, 9.34760650e-04, 2.06613572e-02,
        3.43052171e-01, 2.65682738e-11, 2.44266195e-02, 1.60218516e-03,
        9.81694481e-02]),
 array([1.97760941e-13, 3.78507712e-02, 2.40685996e-02, 2.02946677e-03,
        8.24517274e-05, 8.06366009e-02, 1.38869167e-04, 5.07672805e-02,
      

In [12]:
def longfunction(arr):

    for x in arr:

        yield x

arr = np.arange(10)

jl.Parallel(n_jobs=2)(jl.delayed(longfunction)(arr))

TypeError: cannot unpack non-iterable function object

In [15]:
ho = (jl.delayed(np.sqrt)(i ** 2) for i in range(10))
ho

<generator object <genexpr> at 0x7e769103fbc0>

In [23]:
def test_func(i):

    time.sleep(5)

    return i

val = jl.Parallel(n_jobs=10)(jl.delayed(test_func)(i) for i in range(10))

print(val)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [18]:
time.sleep(5)

In [7]:
[np.sqrt(i ** 2) for i in range(10)]


[np.float64(0.0),
 np.float64(1.0),
 np.float64(2.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(6.0),
 np.float64(7.0),
 np.float64(8.0),
 np.float64(9.0)]